# Load, Clean, Validate, and Feature-Build Guide

Purpose:
- turn the raw Bybit ETHUSDT 15-minute CSV into a consistent OHLC dataset
- inspect data integrity before any labeling or modeling
- engineer a first-pass feature table in a gap-aware way

Main outputs inside this notebook:
- `price_dataframe`: cleaned market data with `timestamp`, OHLC, and `segment_id`
- `features_dataframe`: engineered features saved to `ML v1/data/ETHUSDT_15m_features.parquet`

Reading guide:
1. Load the raw CSV and normalize its schema.
2. Parse timestamps and enforce numeric OHLC values.
3. Identify gaps and build `segment_id` so rolling windows do not cross missing periods.
4. Run integrity and sanity checks.
5. Engineer online-safe features inside each segment.
6. Save the feature table for downstream notebooks.

Important note:
- `segment_id` is part of the data contract: it prevents rolling statistics from leaking across discontinuous stretches of candles.


dataset source: https://www.kaggle.com/datasets/anubhavbhadani142/bybit-ethusdt-historical-data-2021-2025?phase=FinishSSORegistration&returnUrl=%2Fdatasets%2Fanubhavbhadani142%2Fbybit-ethusdt-historical-data-2021-2025%2Fversions%2F1%3Fresource%3Ddownload&SSORegistrationToken=CfDJ8KfhQMrVil5MrZ6RWpmB4eHR_W0RA9HgDeRZeSkQwmo7rVIhCB9j9_u8J_es1X6_ptZY6XZFmpxtwY5RmigiMGWCAT3AtEwY8raOEDngFTzDov-QHVVsIui6yqAQ0H7QygHCIfq-fD4FzjmoYTtt4wSjKfXTMpIoz9m2APx7qhXATfU_aMlFDXulVJ7zZcTQnIF2WsDRKWvU4tSIF_pFMFI5zdNEx63xK2UyA5L78deiBSwe23hH9F-jq0kzv5RLS8fM2D_zCw3pFuEBQPIksHuc4YidQ1fc0w3xUju8pxWQ2eDRZ477vfApVfHcFlAWCtqBX0KIdW7Wy4rP9sE6yanmGqU_&DisplayName=Artem+Khomytskyi

### Imports + path + display options

In [ ]:
# Keep display settings wide because most checks in this notebook are table-based and easier to audit visually.

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

def locate_ml_root(start=None) -> Path:
    start = (Path.cwd() if start is None else Path(start)).resolve()
    for candidate in [start] + list(start.parents):
        if candidate.name == "ML v1" and (candidate / "data").exists():
            return candidate

        ml_root = candidate / "ML v1"
        if (ml_root / "data").exists() and (ml_root / "code").exists():
            return ml_root

    raise FileNotFoundError(
        "Could not locate the 'ML v1' workspace from the current working directory."
    )


ML_ROOT = locate_ml_root()
PROJECT_ROOT = ML_ROOT.parent
DATA_DIR = ML_ROOT / "data"
CONFIG_DIR = ML_ROOT / "configs"
CODE_DIR = ML_ROOT / "code"

RAW_DATA_PATH = DATA_DIR / "BYBIT_ETHUSDT_15m.csv"
FEATURES_OUTPUT_PATH = DATA_DIR / "ETHUSDT_15m_features.parquet"

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

data_path = RAW_DATA_PATH


### Load CSV (safe) + quick preview

In [2]:
price_dataframe = pd.read_csv(data_path)
price_dataframe.head(10)


,Datetime,Open,High,Low,Close,Volume
0,2021-07-05 12:00:00,2196.61,2214.83,2188.94,2208.84,4.2843
1,2021-07-05 12:15:00,2208.84,2219.13,2208.56,2211.99,0.0128
2,2021-07-05 12:30:00,2211.99,2218.00,2209.42,2214.01,0.0137
3,2021-07-05 12:45:00,2214.01,2214.90,2209.76,2210.56,0.0075
4,2021-07-05 13:00:00,2210.56,2226.37,2208.64,2224.07,0.0137
5,2021-07-05 13:15:00,2224.07,2225.42,2216.06,2223.67,0.0086
6,2021-07-05 13:30:00,2223.67,2229.82,2219.39,2227.91,0.0086
7,2021-07-05 13:45:00,2227.91,2234.40,2225.27,2226.72,0.0120
8,2021-07-05 14:00:00,2226.72,2229.02,2219.68,2223.69,0.0106
9,2021-07-05 14:15:00,2223.69,2229.49,2219.96,2219.96,0.0112


### .info() + column names

In [3]:
price_dataframe.info()
price_dataframe.columns.tolist()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139219 entries, 0 to 139218
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  139219 non-null  object 
 1   Open      139219 non-null  float64
 2   High      139219 non-null  float64
 3   Low       139219 non-null  float64
 4   Close     139219 non-null  float64
 5   Volume    139219 non-null  float64
dtypes: float64(5), object(1)
memory usage: 6.4+ MB


['Datetime', 'Open', 'High', 'Low', 'Close', 'Volume']

### Normalize column names + detect timestamp column

In [4]:
def normalize_column_names(columns: list[str]) -> list[str]:
    """
    Normalize column names to snake_case-ish for consistency.

    Args:
        columns: original column names

    Returns:
        Normalized column names
    """
    normalized = []
    for column in columns:
        column_clean = (
            str(column)
            .strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )
        normalized.append(column_clean)
    return normalized


price_dataframe.columns = normalize_column_names(price_dataframe.columns.tolist())
price_dataframe.columns.tolist()


['datetime', 'open', 'high', 'low', 'close', 'volume']

### Parse timestamp (robust)
This tries common timestamp column names and common units (seconds vs milliseconds).

In [5]:
CANDIDATE_TIME_COLUMNS = [
    "timestamp",
    "time",
    "open_time",
    "open_time_ms",
    "start_time",
    "start",
    "datetime",
    "date",
]

def pick_time_column(columns: list[str]) -> str:
    """
    Choose a likely timestamp column from a list of columns.

    Args:
        columns: DataFrame columns

    Returns:
        Selected time column name

    Raises:
        ValueError if no suitable column is found.
    """
    for candidate in CANDIDATE_TIME_COLUMNS:
        if candidate in columns:
            return candidate
    raise ValueError(
        f"Could not find a timestamp column. Available columns: {columns}"
    )


time_column = pick_time_column(price_dataframe.columns.tolist())
time_column


'datetime'

In [6]:
def parse_timestamp_series(series: pd.Series) -> pd.Series:
    """
    Parse a timestamp series into UTC pandas datetime.

    Supports:
    - integer epoch in seconds or milliseconds
    - ISO-like datetime strings

    Args:
        series: timestamp-like series

    Returns:
        UTC datetime series (timezone-aware)
    """
    series_non_null = series.dropna()

    # If numeric, decide seconds vs milliseconds by magnitude.
    if pd.api.types.is_numeric_dtype(series_non_null):
        sample_value = float(series_non_null.iloc[0])
        unit = "ms" if sample_value > 10_000_000_000 else "s"
        return pd.to_datetime(series, unit=unit, utc=True, errors="coerce")

    # Otherwise parse as datetime string
    return pd.to_datetime(series, utc=True, errors="coerce")


price_dataframe["timestamp"] = parse_timestamp_series(price_dataframe[time_column])
price_dataframe[["timestamp"]].head(10)


,timestamp
0,2021-07-05 12:00:00+00:00
1,2021-07-05 12:15:00+00:00
2,2021-07-05 12:30:00+00:00
3,2021-07-05 12:45:00+00:00
4,2021-07-05 13:00:00+00:00
5,2021-07-05 13:15:00+00:00
6,2021-07-05 13:30:00+00:00
7,2021-07-05 13:45:00+00:00
8,2021-07-05 14:00:00+00:00
9,2021-07-05 14:15:00+00:00


### Standardize OHLCV columns (map / rename)
We want canonical: timestamp, open, high, low, close, volume.

In [7]:
CANDIDATE_OHLCV = {
    "open": ["open", "o"],
    "high": ["high", "h"],
    "low": ["low", "l"],
    "close": ["close", "c"],
    "volume": ["volume", "vol", "v", "turnover", "qty"],
}

def find_column(columns: list[str], candidates: list[str]) -> Optional[str]:
    """
    Find the first matching candidate column.

    Args:
        columns: DataFrame columns
        candidates: possible column names for a field

    Returns:
        Matching column name or None
    """
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None


rename_map: dict[str, str] = {}
for canonical_name, candidates in CANDIDATE_OHLCV.items():
    found = find_column(price_dataframe.columns.tolist(), candidates)
    if found is not None and found != canonical_name:
        rename_map[found] = canonical_name

price_dataframe = price_dataframe.rename(columns=rename_map)
sorted(price_dataframe.columns.tolist())


['close', 'datetime', 'high', 'low', 'open', 'timestamp', 'volume']

### Keep only needed columns + enforce numeric types

In [8]:
required_columns = ["timestamp", "open", "high", "low", "close"]
optional_columns = ["volume"]

missing_required = [col for col in required_columns if col not in price_dataframe.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

keep_columns = required_columns + [col for col in optional_columns if col in price_dataframe.columns]
price_dataframe = price_dataframe[keep_columns].copy()

for col in ["open", "high", "low", "close"] + ([c for c in ["volume"] if c in price_dataframe.columns]):
    price_dataframe[col] = pd.to_numeric(price_dataframe[col], errors="coerce")

price_dataframe.head(10)


,timestamp,open,high,low,close,volume
0,2021-07-05 12:00:00+00:00,2196.61,2214.83,2188.94,2208.84,4.2843
1,2021-07-05 12:15:00+00:00,2208.84,2219.13,2208.56,2211.99,0.0128
2,2021-07-05 12:30:00+00:00,2211.99,2218.00,2209.42,2214.01,0.0137
3,2021-07-05 12:45:00+00:00,2214.01,2214.90,2209.76,2210.56,0.0075
4,2021-07-05 13:00:00+00:00,2210.56,2226.37,2208.64,2224.07,0.0137
5,2021-07-05 13:15:00+00:00,2224.07,2225.42,2216.06,2223.67,0.0086
6,2021-07-05 13:30:00+00:00,2223.67,2229.82,2219.39,2227.91,0.0086
7,2021-07-05 13:45:00+00:00,2227.91,2234.40,2225.27,2226.72,0.0120
8,2021-07-05 14:00:00+00:00,2226.72,2229.02,2219.68,2223.69,0.0106
9,2021-07-05 14:15:00+00:00,2223.69,2229.49,2219.96,2219.96,0.0112


### Sort, drop null timestamps, quick .info() again

In [9]:
price_dataframe = price_dataframe.dropna(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

price_dataframe.info()
price_dataframe.head(5)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 139219 entries, 0 to 139218
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype              
---  ------     --------------   -----              
 0   timestamp  139219 non-null  datetime64[ns, UTC]
 1   open       139219 non-null  float64            
 2   high       139219 non-null  float64            
 3   low        139219 non-null  float64            
 4   close      139219 non-null  float64            
 5   volume     139219 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(5)
memory usage: 6.4 MB


,timestamp,open,high,low,close,volume
0,2021-07-05 12:00:00+00:00,2196.61,2214.83,2188.94,2208.84,4.2843
1,2021-07-05 12:15:00+00:00,2208.84,2219.13,2208.56,2211.99,0.0128
2,2021-07-05 12:30:00+00:00,2211.99,2218.00,2209.42,2214.01,0.0137
3,2021-07-05 12:45:00+00:00,2214.01,2214.90,2209.76,2210.56,0.0075
4,2021-07-05 13:00:00+00:00,2210.56,2226.37,2208.64,2224.07,0.0137


### Coverage: min/max dates, row count, basic stats

In [10]:
start_timestamp = price_dataframe["timestamp"].min()
end_timestamp = price_dataframe["timestamp"].max()
row_count = len(price_dataframe)

start_timestamp, end_timestamp, row_count


(Timestamp('2021-07-05 12:00:00+0000', tz='UTC'),
 Timestamp('2025-06-28 20:30:00+0000', tz='UTC'),
 139219)

In [11]:
# Each time gap larger than one 15-minute bar starts a new contiguous segment.

expected_bar_seconds = 15 * 60

timestamp_diff_seconds = (
    price_dataframe["timestamp"]
    .diff()
    .dt.total_seconds()
)

gap_mask = timestamp_diff_seconds > expected_bar_seconds

price_dataframe["segment_id"] = gap_mask.cumsum()


In [12]:
price_dataframe["segment_id"].value_counts().sort_index()

segment_id
0     20219
1    119000
Name: count, dtype: int64

In [13]:
price_dataframe[["open", "high", "low", "close"]].describe()


,open,high,low,close
count,139219.000000,139219.000000,139219.000000,139219.000000
mean,2445.606445,2451.381320,2439.615437,2445.605891
std,853.754550,855.990444,851.359380,853.753193
min,898.830000,904.830000,880.840000,898.830000
25%,1743.020000,1746.250000,1739.205000,1743.020000
50%,2349.510000,2355.460000,2344.210000,2349.530000
75%,3136.350000,3143.445000,3127.995000,3136.350000
max,4845.870000,4863.810000,4842.220000,4845.870000


### Integrity checks (duplicates, monotonicity)

In [14]:
is_monotonic = price_dataframe["timestamp"].is_monotonic_increasing
duplicate_count = int(price_dataframe["timestamp"].duplicated().sum())

is_monotonic, duplicate_count


(True, 0)

In [15]:
if duplicate_count > 0:
    duplicate_rows = price_dataframe.loc[price_dataframe["timestamp"].duplicated(keep=False)].copy()
    duplicate_rows.head(20)


### Check expected frequency (15m) + gaps report

In [16]:
expected_bar_seconds = 15 * 60

timestamp_diff_seconds = (
    price_dataframe["timestamp"]
    .diff()
    .dt.total_seconds()
)

diff_value_counts = (
    timestamp_diff_seconds
    .value_counts(dropna=True)
    .sort_index()
)

diff_value_counts.head(20)


timestamp
900.0       139217
360900.0         1
Name: count, dtype: int64

In [17]:
gap_mask = timestamp_diff_seconds > expected_bar_seconds
gap_count = int(gap_mask.sum())

gap_count


1

In [18]:
if gap_count > 0:
    gap_rows = price_dataframe.loc[gap_mask, ["timestamp"]].copy()

    # Keep timezone-aware dtype by assigning a Series (not .values)
    gap_rows["prev_timestamp"] = price_dataframe["timestamp"].shift(1).loc[gap_mask]

    gap_rows["gap_minutes"] = (
        (gap_rows["timestamp"] - gap_rows["prev_timestamp"])
        .dt.total_seconds()
        / 60.0
    )

    gap_rows.head(30)


### Sanity invariants (OHLC correctness)
This is the “must not be violated” set. For exploration we’ll just count violations and show examples.

In [19]:
open_series = price_dataframe["open"]
high_series = price_dataframe["high"]
low_series = price_dataframe["low"]
close_series = price_dataframe["close"]

violations = {
    "high_lt_max_open_close": (high_series < np.maximum(open_series, close_series)),
    "low_gt_min_open_close": (low_series > np.minimum(open_series, close_series)),
    "high_lt_low": (high_series < low_series),
    "close_le_zero": (close_series <= 0),
    "any_nan_ohlc": price_dataframe[["open", "high", "low", "close"]].isna().any(axis=1),
}

violation_counts = {name: int(mask.sum()) for name, mask in violations.items()}
violation_counts


{'high_lt_max_open_close': 0,
 'low_gt_min_open_close': 0,
 'high_lt_low': 0,
 'close_le_zero': 0,
 'any_nan_ohlc': 0}

In [20]:
# Show a few examples for each violation type (if any)
for name, mask in violations.items():
    count = int(mask.sum())
    if count == 0:
        continue

    print(f"\nViolation: {name} | count={count}")
    display(price_dataframe.loc[mask, ["timestamp", "open", "high", "low", "close"]].head(10))


### Quick “price continuity” sanity (optional)
This is not a “hard invariant”, but catches obvious data glitches.

In [21]:
ret_1 = price_dataframe["close"].pct_change()
ret_stats = ret_1.describe(percentiles=[0.001, 0.01, 0.99, 0.999])

ret_stats


count    139218.000000
mean          0.000008
std           0.003800
min          -0.117031
0.1%         -0.023623
1%           -0.010693
50%           0.000019
99%           0.010504
99.9%         0.022646
max           0.113149
Name: close, dtype: float64

In [22]:
extreme_move_mask = ret_1.abs() > 0.30  # 30% per 15m is suspicious
extreme_count = int(extreme_move_mask.sum())

extreme_count


0

In [23]:
if extreme_count > 0:
    price_dataframe.loc[extreme_move_mask, ["timestamp", "open", "high", "low", "close"]].head(20)


# Feature engineering

In [ ]:
@dataclass(frozen=True)
class FeatureConfig:
    """
    Configuration for feature engineering.
    """
    rolling_windows: tuple[int, ...]
    include_ohlc_for_debug: bool
    output_path: Path


feature_config = FeatureConfig(
    rolling_windows=(20, 50),
    include_ohlc_for_debug=True,
    output_path=FEATURES_OUTPUT_PATH,
)


### Validate input + sort (critical)

In [26]:
ohlc_clean = price_dataframe.copy()

In [27]:
# From this point on, downstream feature code assumes the schema and ordering are already clean.

def validate_ohlc_clean_schema(ohlc_clean: pd.DataFrame) -> None:
    """
    Validate required schema for OHLC clean dataset.

    Args:
        ohlc_clean: cleaned OHLC DataFrame

    Raises:
        ValueError: if required columns are missing
        TypeError: if timestamp is not timezone-aware datetime
    """
    required_columns = {"timestamp", "segment_id", "open", "high", "low", "close"}
    missing_columns = sorted(required_columns - set(ohlc_clean.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    if not pd.api.types.is_datetime64tz_dtype(ohlc_clean["timestamp"]):
        raise TypeError("Column 'timestamp' must be timezone-aware datetime (e.g., UTC).")


validate_ohlc_clean_schema(ohlc_clean)

ohlc_clean = (
    ohlc_clean
    .sort_values(["segment_id", "timestamp"])
    .reset_index(drop=True)
    .copy()
)

ohlc_clean.head(5)


C:\Users\artkh.000\AppData\Local\Temp\ipykernel_30496\1702409234.py:17: DeprecationWarning: is_datetime64tz_dtype is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.DatetimeTZDtype)` instead.
  if not pd.api.types.is_datetime64tz_dtype(ohlc_clean["timestamp"]):


,timestamp,open,high,low,close,volume,segment_id
0,2021-07-05 12:00:00+00:00,2196.61,2214.83,2188.94,2208.84,4.2843,0
1,2021-07-05 12:15:00+00:00,2208.84,2219.13,2208.56,2211.99,0.0128,0
2,2021-07-05 12:30:00+00:00,2211.99,2218.00,2209.42,2214.01,0.0137,0
3,2021-07-05 12:45:00+00:00,2214.01,2214.90,2209.76,2210.56,0.0075,0
4,2021-07-05 13:00:00+00:00,2210.56,2226.37,2208.64,2224.07,0.0137,0


### Helpers (segment-safe rolling)

In [28]:
def groupby_rolling_agg(
    ohlc_clean: pd.DataFrame,
    group_column: str,
    value_column: str,
    window: int,
    agg: str,
) -> pd.Series:
    """
    Compute rolling aggregation within groups, aligned to the original index.

    Args:
        ohlc_clean: input DataFrame sorted by [group_column, timestamp]
        group_column: grouping column name (e.g., 'segment_id')
        value_column: column to aggregate (e.g., 'high', 'low', 'ret_1')
        window: rolling window size
        agg: aggregation name ('max', 'min', 'std')

    Returns:
        A Series aligned with ohlc_clean.index.
    """
    grouped = ohlc_clean.groupby(group_column, sort=False)[value_column]

    if agg == "max":
        result = grouped.rolling(window=window, min_periods=window).max()
    elif agg == "min":
        result = grouped.rolling(window=window, min_periods=window).min()
    elif agg == "std":
        result = grouped.rolling(window=window, min_periods=window).std()
    else:
        raise ValueError(f"Unsupported agg: {agg}")

    return result.reset_index(level=0, drop=True)


### Build Features V0

In [29]:
features_dataframe = pd.DataFrame(
    {
        "timestamp": ohlc_clean["timestamp"],
        "segment_id": ohlc_clean["segment_id"],
    }
)

# --- Returns (segment-safe)
grouped_close = ohlc_clean.groupby("segment_id", sort=False)["close"]

features_dataframe["ret_1"] = grouped_close.pct_change(periods=1)
features_dataframe["ret_3"] = grouped_close.pct_change(periods=3)
features_dataframe["ret_10"] = grouped_close.pct_change(periods=10)

# --- Bar shape (already online-safe)
close_series = ohlc_clean["close"]
open_series = ohlc_clean["open"]
high_series = ohlc_clean["high"]
low_series = ohlc_clean["low"]

features_dataframe["range"] = (high_series - low_series) / close_series
features_dataframe["body"] = (close_series - open_series).abs() / close_series
features_dataframe["upper_wick"] = (high_series - np.maximum(open_series, close_series)) / close_series
features_dataframe["lower_wick"] = (np.minimum(open_series, close_series) - low_series) / close_series

# --- Rolling context (segment-safe)
for window in feature_config.rolling_windows:
    roll_max = groupby_rolling_agg(
        ohlc_clean=ohlc_clean,
        group_column="segment_id",
        value_column="high",
        window=window,
        agg="max",
    )
    roll_min = groupby_rolling_agg(
        ohlc_clean=ohlc_clean,
        group_column="segment_id",
        value_column="low",
        window=window,
        agg="min",
    )
    vol = groupby_rolling_agg(
        ohlc_clean=features_dataframe.assign(ret_1=features_dataframe["ret_1"]),
        group_column="segment_id",
        value_column="ret_1",
        window=window,
        agg="std",
    )

    features_dataframe[f"roll_max_{window}"] = roll_max
    features_dataframe[f"roll_min_{window}"] = roll_min
    features_dataframe[f"dist_to_roll_max_{window}"] = (roll_max - close_series) / close_series
    features_dataframe[f"dist_to_roll_min_{window}"] = (close_series - roll_min) / close_series
    features_dataframe[f"vol_{window}"] = vol

# Optional: keep OHLC for debugging / downstream inspection
if feature_config.include_ohlc_for_debug:
    debug_columns = ["open", "high", "low", "close"]
    if "volume" in ohlc_clean.columns:
        debug_columns.append("volume")

    for col in debug_columns:
        features_dataframe[col] = ohlc_clean[col]

features_dataframe.head(10)


,timestamp,segment_id,ret_1,ret_3,ret_10,range,body,upper_wick,lower_wick,roll_max_20,roll_min_20,dist_to_roll_max_20,dist_to_roll_min_20,vol_20,roll_max_50,roll_min_50,dist_to_roll_max_50,dist_to_roll_min_50,vol_50,open,high,low,close,volume
0,2021-07-05 12:00:00+00:00,0,NaN,NaN,NaN,0.011721,0.005537,0.002712,0.003472,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2196.61,2214.83,2188.94,2208.84,4.2843
1,2021-07-05 12:15:00+00:00,0,0.001426,NaN,NaN,0.004779,0.001424,0.003228,0.000127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2208.84,2219.13,2208.56,2211.99,0.0128
2,2021-07-05 12:30:00+00:00,0,0.000913,NaN,NaN,0.003875,0.000912,0.001802,0.001161,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2211.99,2218.00,2209.42,2214.01,0.0137
3,2021-07-05 12:45:00+00:00,0,-0.001558,0.000779,NaN,0.002325,0.001561,0.000403,0.000362,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2214.01,2214.90,2209.76,2210.56,0.0075
4,2021-07-05 13:00:00+00:00,0,0.006112,0.005461,NaN,0.007972,0.006074,0.001034,0.000863,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2210.56,2226.37,2208.64,2224.07,0.0137
5,2021-07-05 13:15:00+00:00,0,-0.000180,0.004363,NaN,0.004209,0.000180,0.000607,0.003422,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2224.07,2225.42,2216.06,2223.67,0.0086
6,2021-07-05 13:30:00+00:00,0,0.001907,0.007849,NaN,0.004682,0.001903,0.000857,0.001921,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2223.67,2229.82,2219.39,2227.91,0.0086
7,2021-07-05 13:45:00+00:00,0,-0.000534,0.001192,NaN,0.004100,0.000534,0.002915,0.000651,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2227.91,2234.40,2225.27,2226.72,0.0120
8,2021-07-05 14:00:00+00:00,0,-0.001361,0.000009,NaN,0.004200,0.001363,0.001034,0.001803,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2226.72,2229.02,2219.68,2223.69,0.0106
9,2021-07-05 14:15:00+00:00,0,-0.001677,-0.003568,NaN,0.004293,0.001680,0.002613,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2223.69,2229.49,2219.96,2219.96,0.0112


# Build returns, bar-shape, and rolling-context features without letting any calculation cross segment boundaries.

### Quick sanity on NaNs (expected)


In [31]:
nan_counts = features_dataframe.isna().sum().sort_values(ascending=False)
nan_counts.head(20)


vol_50                 100
dist_to_roll_max_50     98
roll_min_50             98
roll_max_50             98
dist_to_roll_min_50     98
vol_20                  40
roll_max_20             38
dist_to_roll_max_20     38
roll_min_20             38
dist_to_roll_min_20     38
ret_10                  20
ret_3                    6
ret_1                    2
high                     0
low                      0
close                    0
open                     0
timestamp                0
segment_id               0
lower_wick               0
dtype: int64

### Save features parquet

In [ ]:
feature_config.output_path.parent.mkdir(parents=True, exist_ok=True)

features_dataframe.to_parquet(feature_config.output_path, index=False)

feature_config.output_path


# Save the engineered feature table as the hand-off artifact for label-building and model notebooks.

### Minimal checks to confirm segment-safety


In [33]:
summary_by_segment = (
    features_dataframe
    .groupby("segment_id", as_index=False)
    .agg(
        start_timestamp=("timestamp", "min"),
        end_timestamp=("timestamp", "max"),
        rows=("timestamp", "count"),
        non_null_ret_1=("ret_1", lambda s: int(s.notna().sum())),
    )
)

summary_by_segment


,segment_id,start_timestamp,end_timestamp,rows,non_null_ret_1
0,0,2021-07-05 12:00:00+00:00,2022-02-01 02:30:00+00:00,20219,20218
1,1,2022-02-05 06:45:00+00:00,2025-06-28 20:30:00+00:00,119000,118999
